In [ ]:
import numpy as np
import pandas as pd

from vis_utils import (
    discover_all_variants, validate_all_files, create_results_table,
    plot_experiment_results, print_latex_table,
)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

START_SEED = 50
NUM_SEEDS = 150

FOLDER_SUFFIX = '_gso'
EXPERIMENT_DATASET = 'gso'

RESULT_DIRS = {
    'Scout_no_decouple': f'../results/{EXPERIMENT_DATASET}/scout_no_decouple/results{FOLDER_SUFFIX}',
    'Knn': f'../results/{EXPERIMENT_DATASET}/knn_script/results{FOLDER_SUFFIX}',
    'Mean': f'../results/{EXPERIMENT_DATASET}/mean_script/results{FOLDER_SUFFIX}',
    'Mlp': f'../results/{EXPERIMENT_DATASET}/mlp_script/results{FOLDER_SUFFIX}',
    'Linear': f'../results/{EXPERIMENT_DATASET}/linear_regression_script/results{FOLDER_SUFFIX}',
    'MF': f'../results/{EXPERIMENT_DATASET}/mf_script/results{FOLDER_SUFFIX}',
    'RouterDC': f'../results/{EXPERIMENT_DATASET}/routerdc_script/results{FOLDER_SUFFIX}',
    'Scout_proper': f'../results/{EXPERIMENT_DATASET}/scout_proper/results{FOLDER_SUFFIX}',
}

FILE_PATTERNS = {key: 'results_{}.txt' for key in RESULT_DIRS}

EXP_METRIC_NAMES = [
    'single_point_exp',
    'full_cost_range_exp',
    'gen_cost_range_exp',
]

# ---------------------------------------------------------------------------
# Discover variants and build metric names
# ---------------------------------------------------------------------------

all_discovered_variants = discover_all_variants(
    RESULT_DIRS, FILE_PATTERNS, EXP_METRIC_NAMES, NUM_SEEDS
)
print(f"Discovered variants: {all_discovered_variants}")

metric_names = []
for exp in EXP_METRIC_NAMES:
    for variant in all_discovered_variants:
        metric_names.append(exp + variant)

print(f"Total metrics: {len(metric_names)}")
print(f"Sample metrics: {metric_names[:10]}")

# ---------------------------------------------------------------------------
# Validate files
# ---------------------------------------------------------------------------

routerdc_metrics = [m for m in metric_names if m.startswith('single_point_exp')]
seeds_to_check = range(START_SEED, START_SEED + NUM_SEEDS)

is_valid = validate_all_files(
    RESULT_DIRS, FILE_PATTERNS, metric_names, seeds_to_check,
    method_metric_names={'RouterDC': routerdc_metrics},
)
if not is_valid:
    raise ValueError("Cannot proceed with missing entries!")

# ---------------------------------------------------------------------------
# Build results tables
# ---------------------------------------------------------------------------

print("=" * 80)
print("RESULTS ANALYSIS")
print("=" * 80)

tables = {}
for method_name, result_dir in RESULT_DIRS.items():
    print(f"\n{'=' * 80}")
    print(f"{method_name} Results")
    print('=' * 80)

    file_pattern = FILE_PATTERNS[method_name]
    table = create_results_table(
        method_name, result_dir, file_pattern, metric_names,
        seeds=range(START_SEED, START_SEED + NUM_SEEDS),
    )
    tables[method_name] = table

    print(f"\nTable with seeds as columns (showing first 10 seeds):")
    print(table.iloc[:, :10].to_string())
    print(f"\nMean and Standard Error:")
    print(table[['mean', 'std_err']].to_string())

# ---------------------------------------------------------------------------
# Cross-method comparison
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARISON ACROSS ALL METHODS")
print("=" * 80)

comparison_data = []
for method_name, table in tables.items():
    method_df = table[['mean', 'std_err']].copy()
    method_df.columns = [f'{method_name}_mean', f'{method_name}_std_err']
    comparison_data.append(method_df)

comparison_table = pd.concat(comparison_data, axis=1)
print("\n" + comparison_table.to_string())

# ---------------------------------------------------------------------------
# Best method per metric
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("BEST METHOD FOR EACH METRIC (by mean)")
print("=" * 80)

all_methods = list(tables.keys())

for metric in metric_names:
    method_means = {
        method: tables[method].loc[metric, 'mean'] for method in tables
    }

    if 'regret' in metric or 'exp' in metric or 'Exp' in metric:
        best_method = min(method_means, key=method_means.get)
    else:
        best_method = max(method_means, key=method_means.get)

    print(f"{metric:25s}: {best_method:15s} (value: {method_means[best_method]:.6f})")

print("\n" + "=" * 80)
print("Analysis complete!")
print("=" * 80)

In [ ]:
plot_experiment_results(tables, EXP_METRIC_NAMES)

In [ ]:
# ---------------------------------------------------------------------------
# Summary table: normalized by Mean method
# ---------------------------------------------------------------------------

all_methods = list(tables.keys())
summary = {}

MEANPASS_ENTRIES = {
    'MP-Hunyuan': '-hunyuan',
    'MP-InstantMesh': '-instant_mesh',
    'MP-Trellis': '-trellis',
    'MP-TripoSR': '-triposr',
}

for exp in EXP_METRIC_NAMES:
    if 'Mean' not in tables or exp not in tables['Mean'].index:
        print(f"Warning: 'Mean' method missing for {exp}, skipping normalization.")
        continue

    mean_baseline = tables['Mean'].loc[exp, 'mean']

    for method in all_methods:
        if method not in tables or exp not in tables[method].index:
            continue

        mean_val = tables[method].loc[exp, 'mean']
        std_err_val = tables[method].loc[exp, 'std_err']
        if np.isnan(mean_val):
            continue

        norm_mean = mean_val / mean_baseline
        norm_std_err = std_err_val / mean_baseline

        if method not in summary:
            summary[method] = {}
        summary[method][exp] = (norm_mean, norm_std_err)

    # MeanPass variants
    for label, variant in MEANPASS_ENTRIES.items():
        metric_name = exp + variant
        if metric_name not in tables['Mean'].index:
            continue
        mean_val = tables['Mean'].loc[metric_name, 'mean']
        std_err_val = tables['Mean'].loc[metric_name, 'std_err']
        if np.isnan(mean_val):
            continue
        if label not in summary:
            summary[label] = {}
        summary[label][exp] = (mean_val / mean_baseline, std_err_val / mean_baseline)

display_order = [
    'RouterDC',
    'Mlp',
    'MF',
    'Knn',
    'Linear',
    'Scout_no_decouple',
    'Scout_proper',
    'Mean',
    'MP-Hunyuan',
    'MP-InstantMesh',
    'MP-Trellis',
    'MP-TripoSR',
]

print('Normalized by Input Agnostic (Mean):')
print('=' * (25 + 17 * len(EXP_METRIC_NAMES)))
for method in display_order:
    if method not in summary:
        continue
    label = 'Input Agnostic' if method == 'Mean' else method
    if method == 'Mean':
        vals = [f"{summary[method][exp][0]:.4f}" if exp in summary[method] else '--' for exp in EXP_METRIC_NAMES]
    else:
        vals = [f"{summary[method][exp][0]:.4f}±{summary[method][exp][1]:.4f}" if exp in summary[method] else '--' for exp in EXP_METRIC_NAMES]
    print(f"{label:<25s} {' '.join(f'{v:>16s}' for v in vals)}")